# 02 · E4: comparison of pandas, DuckDB and Spark (RQ4)

**RQ4:** Which of pandas, DuckDB and PySpark is most suitable for this workload, as judged by run time, memory use and throughput under varying data size, number of cores and file format?

**Workload** (identical in all three tools, `src/e4.py`): read the raw 2023–2025 WYS speed-bin records for the selected months, aggregate them to sign-hour speed metrics (volume, mean, 85th percentile, share speeding), join the result with the weather table and with the collisions matched to each sign, and write Parquet.

**Design:** 3 tools × 25/50/75/100 % of the data (complete months) × 1/2/4/8/16 cores × CSV/Parquet, 3 runs each, median reported. pandas reads in chunks on a single thread. Each run is a fresh process; peak memory includes the Spark JVM.

**Fairness:** same machine, same input files and outputs, no other heavy processes. Output row counts and column totals are compared across tools.

**Selection rules** (Section VI-B): (1) must finish the full workload without running out of memory, with the same results; (2) lowest median run time at 100 %; (3) if within ~20 %, the tool whose run time grows more slowly from 25 % to 100 %; (4) otherwise the easier tool to maintain.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.spark_session import load_config, path, get_spark

cfg = load_config()
FIG, TAB = path(cfg, "figures"), path(cfg, "tables")
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
PROC = path(cfg, "processed"); PROC.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})

## 1. Shared inputs (not timed)
The spatial match uses the 150 m buffer from the proposal.

In [ ]:
from src import e4, spatial, wys
coll = pd.read_parquet(PROC / "collisions.parquet")
wx = pd.read_parquet(PROC / "weather_hourly.parquet")
locations = pd.read_parquet(PROC / "wys_locations.parquet")
spark = get_spark(cfg); spark.sparkContext.setLogLevel("ERROR")
m = spatial.match_collisions_to_signs(spark, coll, locations, cfg["spatial"]["buffer_m"]).toPandas()
m = m.merge(coll[["collision_id", "date", "hour"]], on="collision_id")
mh = m.groupby(["sign_id", "date", "hour"]).size().rename("collisions").reset_index()
spark.stop()
e4.write_side_tables(cfg, locations, wx, mh)
for f in (0.25, 0.5, 0.75, 1.0):
    print(f, "->", len(wys.month_list(cfg, f)), "months:", wys.month_list(cfg, f)[:4], "...")

## 2. Run the experiment

The full design is 264 runs and takes several hours. Run it from a terminal so that the notebook kernel does not compete for CPU or memory:

```bash
python scripts/run_e4.py --pilot     # one-month pilot first (about 10 minutes)
python scripts/run_e4.py             # full design, resumable
```

Results are appended to `results/e4/runs.csv`.

In [ ]:
runs = pd.read_csv(path(cfg, "e4", "runs.csv"))
hw = pd.read_json(path(cfg, "e4", "hardware.json"), typ="series")
print(hw.to_string())
ok = runs[runs["status"] == "ok"]
print(f"{len(ok)} successful runs, {len(runs) - len(ok)} failed or timed out")
runs[runs["status"] != "ok"][["tool", "cores", "fraction", "fmt", "repeat", "status", "peak_mem_gb"]]

## 3. Correctness: all tools must produce the same output

In [ ]:
chk = ok.groupby(["fraction"])[["rows_out", "sum_volume", "sum_collisions", "mean_p85"]].nunique()
display(chk)
assert (chk[["rows_out", "sum_volume", "sum_collisions"]] == 1).all().all(), "tools disagree"
print("identical row counts and totals across all tools, cores and formats")

## 4. Median results

In [ ]:
med = (ok.groupby(["tool", "fmt", "cores", "fraction"])
         .agg(total_s=("total_s", "median"), workload_s=("workload_s", "median"),
              spark_startup_s=("spark_startup_s", "median"), peak_mem_gb=("peak_mem_gb", "median"),
              rows_in=("rows_in", "first"))
         .reset_index())
med["rows_per_s"] = med["rows_in"] / med["total_s"]
base = med[med["cores"] == 1].set_index(["tool", "fmt", "fraction"])["total_s"]
med["speedup"] = base.reindex(med.set_index(["tool", "fmt", "fraction"]).index).to_numpy() / med["total_s"]
med.to_csv(TAB / "e4_median_results.csv", index=False)

best = med.loc[med.groupby(["tool", "fmt", "fraction"])["total_s"].idxmin()]
table = best.pivot_table(index=["tool", "fmt"], columns="fraction", values="total_s").round(1)
table.columns = [f"{int(c*100)}% (s)" for c in table.columns]
table.to_csv(TAB / "e4_runtime_by_size.csv")
table

In [ ]:
full = med[med["fraction"] == 1.0]
summary = (full.loc[full.groupby(["tool", "fmt"])["total_s"].idxmin(),
                    ["tool", "fmt", "cores", "total_s", "spark_startup_s", "peak_mem_gb", "rows_per_s"]]
           .sort_values("total_s").round(2))
summary.to_csv(TAB / "e4_summary_full_data.csv", index=False)
summary

## 5. Figures

In [ ]:
colors = {"pandas": "#8c8c8c", "duckdb": "#DD8452", "spark": "#4C72B0"}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for (tool, fmt), g in best.groupby(["tool", "fmt"]):
    axes[0].plot(g["fraction"] * 100, g["total_s"], marker="o", color=colors[tool],
                 ls="-" if fmt == "parquet" else "--", label=f"{tool} ({fmt})")
axes[0].set_xlabel("share of data (%)"); axes[0].set_ylabel("median run time (s)"); axes[0].set_title("Run time vs data size (best cores)")
axes[0].legend(frameon=False, fontsize=7)
sc = med[(med["fraction"] == 1.0) & (med["fmt"] == "parquet") & (med["tool"] != "pandas")]
for tool, g in sc.groupby("tool"):
    axes[1].plot(g["cores"], g["speedup"], marker="o", color=colors[tool], label=tool)
axes[1].plot([1, 16], [1, 16], color="black", lw=0.6, ls=":", label="linear")
axes[1].set_xscale("log", base=2); axes[1].set_yscale("log", base=2)
axes[1].set_xlabel("cores"); axes[1].set_ylabel("speed-up vs 1 core"); axes[1].set_title("Scaling, 100% Parquet"); axes[1].legend(frameon=False, fontsize=7)
mem = full.loc[full.groupby(["tool", "fmt"])["total_s"].idxmin()]
axes[2].bar([f"{t}\n{f}" for t, f in zip(mem["tool"], mem["fmt"])], mem["peak_mem_gb"],
            color=[colors[t] for t in mem["tool"]])
axes[2].set_ylabel("peak memory (GB)"); axes[2].set_title("Peak memory, 100% data")
fig.savefig(FIG / "fig_e4_performance.png"); plt.show()

In [ ]:
# D2 Section 6: two configurations of the same operation (Spark, Parquet, all cores): 50% vs 100%
cores_max = med["cores"].max()
two = med[(med["tool"] == "spark") & (med["fmt"] == "parquet") & (med["cores"] == cores_max) & med["fraction"].isin([0.5, 1.0])]
two = two[["fraction", "rows_in", "total_s", "workload_s", "spark_startup_s", "peak_mem_gb", "rows_per_s"]].round(2)
two.to_csv(TAB / "e4_spark_50_vs_100.csv", index=False)
display(two)
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["50% of data", "100% of data"], two["workload_s"], color="#4C72B0", label="workload")
ax.bar(["50% of data", "100% of data"], two["spark_startup_s"], bottom=two["workload_s"], color="#A1C9F4", label="JVM start-up")
ax.set_ylabel("median time (s)"); ax.set_title(f"Spark local[{cores_max}], Parquet"); ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "fig_spark_50_vs_100.png"); plt.show()

## 6. Tool selection

In [ ]:
completed = set(ok[ok["fraction"] == 1.0]["tool"])
rule1 = [t for t in cfg["e4"]["tools"] if t in completed]
full_best = full.groupby("tool")["total_s"].min().loc[rule1].sort_values()
growth = best[best["fmt"] == "parquet"].pivot_table(index="tool", columns="fraction", values="total_s")
growth = (growth[1.0] / growth[0.25]).rename("time_100_over_25")
print("Rule 1 (completed 100%, same results):", rule1)
print("Rule 2 (median run time at 100%):"); print(full_best.round(1).to_string())
first, second = full_best.index[0], full_best.index[1]
gap = full_best.iloc[1] / full_best.iloc[0] - 1
if gap < 0.20:
    print(f"Rule 3: {first} and {second} differ by {gap:.0%} (<20%), compare growth 25%->100%:"); print(growth.round(2).to_string())
    selected = growth.loc[[first, second]].idxmin()
else:
    selected = first
print("\nSelected processing tool:", selected)

**H4** (Spark is slower on small inputs but catches up as data grows; Parquet is faster than CSV for every tool; extra cores give less than proportional speed-up): check each part against the tables and figures above.

Whichever tool is selected, the pipeline in notebook 03 demonstrates the processing steps with Spark DataFrame operations, and the same aggregation and joins were run in all three tools here.